# 01 — Preprocessing: Cleansing & Tokenizing Soal LeetCode

**Tujuan:** mengubah hasil scraping (`leetcode_raw_data.csv`) menjadi satu file bersih (`data_clean.csv`) yang dipakai oleh semua notebook berikutnya.

**Input:** `leetcode_raw_data.csv` (kolom: `id, title, slug, difficulty, is_paid_only, category, raw_text`)

**Output:**
- `data_clean.csv` — satu baris per soal, diurutkan berdasarkan `id`
- `data_clean_meta.json` — catatan untuk paper (tanggal scraping, jumlah data, daftar stopword tambahan)

**Tiga "view" teks yang dihasilkan** (alasannya ada di rancangan eksperimen):

| Kolom | Isi | Dipakai oleh |
|---|---|---|
| `text_light` | HTML entity dibereskan, spasi dirapikan. Huruf besar/kecil, angka, simbol, dan stopword **dipertahankan** | SBERT, CodeBERT (Notebook 04) |
| `text_clean` | lowercase, hanya huruf, tokenisasi, stopword dibuang | TF-IDF, GloVe, FastText |
| `text_clean_digits` | sama dengan `text_clean`, tetapi **angka dipertahankan** | Ablasi angka (Notebook 07) |

Tahapan: 1. Import & konfigurasi → 2. Load → 3. Filter → 4. Tiga view teks → 5. Contoh before/after → 6. Sanity check → 7. Simpan → 8. Analisis singkat

## 1. Import & Konfigurasi

In [3]:
import html
import json
import os
import re
from collections import Counter
from datetime import datetime

import pandas as pd
import nltk

for pkg in ['punkt', 'punkt_tab', 'stopwords']:
    nltk.download(pkg, quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# --- Konfigurasi path (ubah sesuai lokasi file di komputermu) ---
RAW_PATH = r'D:\TrioSetiawan\course\kuliah\s3\text_mining\uts_leetcode\scraping\scraping_outputs\leetcode_raw_data.csv'
OUT_PATH = r'D:\TrioSetiawan\course\kuliah\s3\text_mining\uts_leetcode\preprocessing\preprocessing_output\data_clean.csv'
META_PATH = r'D:\TrioSetiawan\course\kuliah\s3\text_mining\uts_leetcode\preprocessing\preprocessing_output\data_clean_meta.json'

# Kata boilerplate khas soal LeetCode (sama dengan versi sebelumnya)
EXTRA_STOPWORDS = {
    'example', 'input', 'output', 'explanation', 'constraints',
    'return', 'given', 'follow', 'note', 'write', 'function'
}

## 2. Load Dataset

Tanggal pengambilan data diambil dari tanggal modifikasi file raw. Rubrik UTS meminta *waktu pengambilan data* di Bab III-A, jadi catat nilainya.

In [4]:
df_raw = pd.read_csv(RAW_PATH)
scrape_date = datetime.fromtimestamp(os.path.getmtime(RAW_PATH)).strftime('%Y-%m-%d')

print('Jumlah baris raw     :', len(df_raw))
print('Tanggal file (scrape):', scrape_date)
print('Kolom                :', list(df_raw.columns))
df_raw.head()

Jumlah baris raw     : 4047
Tanggal file (scrape): 2026-09-10
Kolom                : ['id', 'title', 'slug', 'difficulty', 'is_paid_only', 'category', 'raw_text']


,id,title,slug,difficulty,is_paid_only,category,raw_text
0,1,Two Sum,two-sum,Easy,False,Array; Hash Table,You are given an array of integers nums and an...
1,2,Add Two Numbers,add-two-numbers,Medium,False,Linked List; Math; Recursion,You are given two non-empty linked lists repre...
2,3,Longest Substring Without Repeating Characters,longest-substring-without-repeating-characters,Medium,False,Hash Table; String; Sliding Window,"Given a string s , find the length of the long..."
3,4,Median of Two Sorted Arrays,median-of-two-sorted-arrays,Hard,False,Array; Binary Search; Divide and Conquer,Given two sorted arrays nums1 and nums2 of siz...
4,5,Longest Palindromic Substring,longest-palindromic-substring,Medium,False,Two Pointers; String; Dynamic Programming; Man...,"Given a string s , return the longest palindro..."


## 3. Filter Data

Soal premium tidak punya teks (`raw_text` kosong/NaN), jadi dibuang. Ini perlu disebut sebagai keterbatasan di paper (*selection bias*: hanya soal gratis yang dianalisis).

In [5]:
n_raw = len(df_raw)
n_premium = int(df_raw['is_paid_only'].sum())

df = df_raw.dropna(subset=['raw_text']).copy()
df = df[df['raw_text'].str.strip() != ''].copy()

# Soal tanpa tag topik diberi string kosong supaya tidak NaN di tahap berikutnya
df['category'] = df['category'].fillna('')

n_used = len(df)
print('Total raw                 :', n_raw)
print('Soal premium (is_paid_only):', n_premium)
print('Dipakai (punya teks)      :', n_used)
print('Soal tanpa tag topik      :', int((df['category'] == '').sum()))

# Pastikan kunci unik (dipakai untuk join antar notebook)
assert df['id'].is_unique, 'id tidak unik'
assert df['slug'].is_unique, 'slug tidak unik'
assert set(df['difficulty'].unique()) <= {'Easy', 'Medium', 'Hard'}, 'label difficulty tak terduga'

Total raw                 : 4047
Soal premium (is_paid_only): 783
Dipakai (punya teks)      : 3256
Soal tanpa tag topik      : 80


## 4. Tiga View Teks

Semua view berasal dari `text_light`. Fungsi-fungsi dibuat terpisah supaya mudah dijelaskan di Bab III-C.

In [6]:
def make_light(text):
    """View ringan: bereskan HTML entity & spasi, tidak menghapus apa pun yang bermakna."""
    text = html.unescape(str(text)).replace('\xa0', ' ')
    return re.sub(r'\s+', ' ', text).strip()


def make_clean(text, keep_digits=False):
    """View bersih: lowercase, buang simbol (dan angka jika keep_digits=False)."""
    text = str(text).lower()
    pattern = r'[^a-z0-9\s]' if keep_digits else r'[^a-z\s]'
    text = re.sub(pattern, ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


stop_words = set(stopwords.words('english')) | EXTRA_STOPWORDS


def tokenize(text):
    return word_tokenize(text)


def remove_stopwords(tokens, keep_digits=False):
    """Buang stopword dan token 1 karakter (token angka 1 karakter dipertahankan bila keep_digits)."""
    return [t for t in tokens
            if t not in stop_words and (len(t) > 1 or (keep_digits and t.isdigit()))]

In [7]:
# View 1: light
df['text_light'] = df['raw_text'].apply(make_light)

# View 2: clean (tanpa angka) -- identik dengan pipeline versi sebelumnya
clean_str = df['text_light'].apply(make_clean)
tokens_all = clean_str.apply(tokenize)
df['n_tokens_before'] = tokens_all.apply(len)
tokens_clean = tokens_all.apply(remove_stopwords)
df['n_tokens_after'] = tokens_clean.apply(len)
df['text_clean'] = tokens_clean.apply(' '.join)

# View 3: clean dengan angka (untuk ablasi)
digits_tokens = (df['text_light']
                 .apply(lambda t: make_clean(t, keep_digits=True))
                 .apply(tokenize)
                 .apply(lambda toks: remove_stopwords(toks, keep_digits=True)))
df['text_clean_digits'] = digits_tokens.apply(' '.join)

df[['id', 'title', 'n_tokens_before', 'n_tokens_after']].head(10)

,id,title,n_tokens_before,n_tokens_after
0,1,Two Sum,102,43
1,2,Add Two Numbers,103,46
2,3,Longest Substring Without Repeating Characters,93,38
3,4,Median of Two Sorted Arrays,69,32
4,5,Longest Palindromic Substring,41,17
5,6,Zigzag Conversion,126,48
6,7,Reverse Integer,56,22
7,8,String to Integer (atoi),315,145
8,9,Palindrome Number,87,32
9,10,Regular Expression Matching,138,60


## 5. Contoh Before / After

Bahan untuk tabel/kotak contoh di Bab III-C (*"contoh teks sebelum dan sesudah preprocessing"*). Pilih satu contoh yang mengandung angka batasan supaya terlihat apa yang hilang di `text_clean`.

In [8]:
sample = df[df['id'].isin([1, 2, 4])] if df['id'].isin([1, 2, 4]).any() else df.head(3)

for _, r in sample.iterrows():
    print('=' * 100)
    print(f"[{r['id']}] {r['title']}  ({r['difficulty']}; {r['category']})")
    print('\nRAW / LIGHT :', r['text_light'][:350])
    print('\nCLEAN       :', r['text_clean'][:350])
    print('\nCLEAN+DIGITS:', r['text_clean_digits'][:350])

[1] Two Sum  (Easy; Array; Hash Table)

RAW / LIGHT : You are given an array of integers nums and an integer target , return indices of the two numbers such that they add up to target . You may assume that each input would have exactly one solution , and you may not use the same element twice. You can return the answer in any order. Example 1: Input: nums = [2,7,11,15], target = 9 Output: [0,1] Explan

CLEAN       : array integers nums integer target indices two numbers add target may assume would exactly one solution may use element twice answer order nums target nums nums nums target nums target nums length nums target one valid answer exists come algorithm less time complexity

CLEAN+DIGITS: array integers nums integer target indices two numbers add target may assume would exactly one solution may use element twice answer order 1 nums 2 7 11 15 target 9 0 1 nums 0 nums 1 9 0 1 2 nums 3 2 4 target 6 1 2 3 nums 3 3 target 6 0 1 2 nums length 10 4 10 9 nums 10 9 10 9 target 10 9 one va

## 6. Sanity Check

In [9]:
# Tidak boleh ada teks kosong di ketiga view
for col in ['text_light', 'text_clean', 'text_clean_digits']:
    n_empty = int((df[col].str.strip() == '').sum())
    print(f'{col:18s} kosong: {n_empty}')
    assert n_empty == 0, f'{col} punya teks kosong'

print()
print(df['n_tokens_before'].describe().round(1).to_string())
print()
print('Proporsi token terbuang (stopword & simbol):',
      round(1 - df['n_tokens_after'].sum() / df['n_tokens_before'].sum(), 3))

# Soal dengan teks paling pendek -- cek manual apakah wajar
df.nsmallest(5, 'n_tokens_after')[['id', 'title', 'n_tokens_before', 'n_tokens_after']]

text_light         kosong: 0
text_clean         kosong: 0
text_clean_digits  kosong: 0

count    3256.0
mean      175.8
std        87.7
min        21.0
25%       113.0
50%       159.0
75%       222.0
max       715.0

Proporsi token terbuang (stopword & simbol): 0.537


,id,title,n_tokens_before,n_tokens_after
21,22,Generate Parentheses,26,7
828,829,Consecutive Numbers Sum,37,7
1555,1556,Thousand Separator,27,7
49,50,"Pow(x, n)",48,8
58,59,Spiral Matrix II,30,8


## 7. Simpan Hasil

Diurutkan berdasarkan `id` (numerik). **Semua notebook berikutnya wajib memakai urutan ini** supaya baris `.npy` fitur selalu sejajar dengan label dan split.

In [10]:
df = df.sort_values('id', key=lambda s: pd.to_numeric(s, errors='coerce')).reset_index(drop=True)

cols = ['id', 'title', 'slug', 'difficulty', 'category',
        'raw_text', 'text_light', 'text_clean', 'text_clean_digits',
        'n_tokens_before', 'n_tokens_after']
df[cols].to_csv(OUT_PATH, index=False)

meta = {
    'scrape_date': scrape_date,
    'n_raw': n_raw,
    'n_premium': n_premium,
    'n_used': n_used,
    'extra_stopwords': sorted(EXTRA_STOPWORDS),
    'difficulty_counts': df['difficulty'].value_counts().to_dict(),
    'tokens_before_mean': round(float(df['n_tokens_before'].mean()), 1),
    'tokens_after_mean': round(float(df['n_tokens_after'].mean()), 1),
}
with open(META_PATH, 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

# Verifikasi: baca ulang dan cek konsistensi
chk = pd.read_csv(OUT_PATH)
assert len(chk) == n_used
assert chk[['text_light', 'text_clean', 'text_clean_digits']].notna().all().all()
print(f'Tersimpan: {OUT_PATH} ({len(chk)} baris), {META_PATH}')
print(json.dumps(meta, indent=2, ensure_ascii=False))

Tersimpan: D:\TrioSetiawan\course\kuliah\s3\text_mining\uts_leetcode\preprocessing\preprocessing_output\data_clean.csv (3256 baris), D:\TrioSetiawan\course\kuliah\s3\text_mining\uts_leetcode\preprocessing\preprocessing_output\data_clean_meta.json
{
  "scrape_date": "2026-09-10",
  "n_raw": 4047,
  "n_premium": 783,
  "n_used": 3256,
  "extra_stopwords": [
    "constraints",
    "example",
    "explanation",
    "follow",
    "function",
    "given",
    "input",
    "note",
    "output",
    "return",
    "write"
  ],
  "difficulty_counts": {
    "Medium": 1645,
    "Easy": 833,
    "Hard": 778
  },
  "tokens_before_mean": 175.8,
  "tokens_after_mean": 81.4
}


## 8. Analisis Singkat

Bahan awal untuk pembahasan RQ4 (kenapa difficulty sulit diklasifikasikan): apakah kosakata antar level benar-benar mirip, dan seberapa besar porsi angka yang terbuang di `text_clean`.

In [11]:
all_tokens = [t for s in df['text_clean'] for t in s.split()]
print('Top 20 kata (seluruh soal):')
print(Counter(all_tokens).most_common(20))

print()
for level in ['Easy', 'Medium', 'Hard']:
    toks = [t for s in df.loc[df['difficulty'] == level, 'text_clean'] for t in s.split()]
    print(f'--- Top 10 kata level {level} ---')
    print(Counter(toks).most_common(10))
    print()

Top 20 kata (seluruh soal):
[('nums', 9969), ('length', 6028), ('array', 4118), ('number', 3308), ('integer', 3168), ('string', 2276), ('one', 2253), ('two', 2164), ('sum', 1741), ('node', 1737), ('index', 1654), ('value', 1569), ('answer', 1475), ('elements', 1392), ('grid', 1348), ('first', 1333), ('maximum', 1324), ('time', 1289), ('arr', 1279), ('integers', 1211)]

--- Top 10 kata level Easy ---
[('nums', 2416), ('length', 1146), ('array', 832), ('number', 721), ('integer', 653), ('string', 582), ('id', 495), ('two', 451), ('one', 392), ('table', 389)]

--- Top 10 kata level Medium ---
[('nums', 4806), ('length', 3028), ('array', 2136), ('number', 1722), ('integer', 1635), ('one', 1232), ('two', 1131), ('string', 1127), ('node', 1005), ('index', 893)]

--- Top 10 kata level Hard ---
[('nums', 2747), ('length', 1854), ('array', 1150), ('integer', 880), ('number', 865), ('edges', 655), ('one', 629), ('two', 582), ('node', 582), ('string', 567)]



In [12]:
# Porsi token angka pada view dengan angka -- seberapa banyak informasi (batasan) yang hilang di text_clean
dig_tokens = [t for s in df['text_clean_digits'] for t in s.split()]
share_digit = sum(t.isdigit() for t in dig_tokens) / len(dig_tokens)
print(f'Porsi token angka di text_clean_digits: {share_digit:.1%}')

# Per level difficulty
for level in ['Easy', 'Medium', 'Hard']:
    toks = [t for s in df.loc[df['difficulty'] == level, 'text_clean_digits'] for t in s.split()]
    print(f'{level:7s}: {sum(t.isdigit() for t in toks) / len(toks):.1%}')

Porsi token angka di text_clean_digits: 37.1%
Easy   : 35.1%
Medium : 37.0%
Hard   : 38.5%
